#!pip install -U ndvi2gif

Hay que revisar el indice de agua nuevo que da valores raros, ver lo de sar y generar un prodcuto nuevo de tendencia de cada pixel!!

Luego los md (readme, changelog, contribution) y sacar la version nueva y pensar en hacer ya lo de JOSS!

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="geemap.conversion")

In [2]:
#!pip install geemap
#!pip install ndvi2gif
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Para ver el entorno conda específico
import os
print(f"CONDA_DEFAULT_ENV: {os.environ.get('CONDA_DEFAULT_ENV', 'No conda env')}")

Python executable: /home/diego/miniconda3/envs/ndvi2gif_clean/bin/python
Python version: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12) 
[GCC 13.3.0]
CONDA_DEFAULT_ENV: ndvi2gif_clean


In [3]:
import geemap
import ee
from ndvi2gif import NdviSeasonality

In [4]:
ee.Authenticate()
ee.Initialize(project='ee-digdgeografo')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [ ]:
help(NdviSeasonality.get_export)

In [ ]:
from ndvi2gif import NdviSeasonality
import inspect

# Verificar que los parámetros están en la firma
print("Parámetros del __init__:")
print(inspect.signature(NdviSeasonality.__init__))

# Crear instancia con debug
print("\nCreando instancia...")
try:
    s2_test = NdviSeasonality(sat='S2', index='avi')
    print("cloud_filter:", hasattr(s2_test, 'cloud_filter'))
    print("max_cloud_cover:", hasattr(s2_test, 'max_cloud_cover'))
except AttributeError as e:
    print(f"Error: {e}")
    # Verificar qué atributos sí tiene
    temp = NdviSeasonality.__new__(NdviSeasonality)
    print("\nAtributos disponibles antes de _setup_satellite_collections:")
    print([attr for attr in dir(temp) if not attr.startswith('_')])

In [ ]:
# Test rápido
from ndvi2gif import NdviSeasonality

# Test S3
s2_test = NdviSeasonality(sat='S2', index='avi')
print(s2_test.get_available_indices('S2'))

In [ ]:
# Count cloud/non cloud images
# Test con filtro activado (estricto)
proc_strict = NdviSeasonality(
    roi=roi,  # tu área de estudio
    sat='S2',
    index='ndvi',
    periods=12,
    start_year=2020,
    end_year=2021,
    cloud_filter=True,
    max_cloud_cover=10  # Solo 10% de nubes
)

# Test sin filtro
proc_no_filter = NdviSeasonality(
    roi=roi,
    sat='S2',
    index='ndvi',
    periods=12,
    start_year=2020,
    end_year=2021,
    cloud_filter=False
)

# Con filtro estricto
collection, df_long = proc_strict.get_year_composite(
    return_counts=True,
    count_valid_pixels=False,
    count_mode="unique_dates",
    return_df=True,        # <- activa DataFrame
    df_pivot=False         # <- largo
)

# Sin filtro
collection, df_wide = proc_strict.get_year_composite(
    return_counts=True,
    count_valid_pixels=False,
    count_mode="unique_dates",
    return_df=True,
    df_pivot=True          # <- ancho (año x periodos)
)

# Mostrar resultados
print(df_long)
#print(df_wide)

In [5]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

We utilize the map drawing tools to create a roi

In [ ]:
# Doñana NDVI Stats
myclass = NdviSeasonality(roi='deimsid:https://deims.org/bcbc866c-3f4f-47a8-bbbc-0a93df6de7b2', sat='S2', 
                          key='perc_95', periods=12,start_year=2019, end_year=2020, index='ndvi')
Landsat = myclass.get_year_composite()
vizParams = {'bands': ['february', 'april', 'july'], 'min': 0.1, 'max':0.8}

Map.addLayer(Landsat, vizParams, 'Max')

In [ ]:
roi = Map.draw_last_feature

#### Mediana MNDWI Landsat

In [ ]:
# Mediana MNDWI de toda la serie Landsat (L4-5-7-8-9 fusionados)
proc = NdviSeasonality(
    roi='deimsid:https://deims.org/bcbc866c-3f4f-47a8-bbbc-0a93df6de7b2',  # Doñana
    sat='Landsat',          # Landsat merged (1982–presente)
    index='mndwi',          # índice MNDWI está soportado
    key='max',       # estadístico por periodo 
    periods=1,              # un único periodo por año
    start_year=2008,        # (o 1982 si quieres forzar al máximo)
    end_year=2011,          # exclusivo => llega hasta 2024
    cloud_filter=True,
    max_cloud_cover=10      # ajusta si lo necesitas
)

# 1 imagen por año (banda 'p1'); después hacemos la mediana interanual
coll = proc.get_year_composite()
mndwi_alltime_median = coll.max()

# Visualización (rango típico MNDWI ~[-1, 1]; agua suele ser positivo)
viz = {'min': 0.2, 'max': 0.6}
Map.addLayer(mndwi_alltime_median, viz, 'Landsat MNDWI Max (2008-2010)')

#### Winter max

In [6]:
# Máxima MNDWI de INVIERNO (Landsat, toda la serie histórica)
roi_landsat = 'wrs:202,034'

proc = NdviSeasonality(
    roi= roi_landsat, # 'deimsid:https://deims.org/bcbc866c-3f4f-47a8-bbbc-0a93df6de7b2',  # Doñana
    sat='Landsat',
    index='mndwi',
    key='percentile',         # dentro de cada invierno, toma el máximo
    percentile = 95,
    periods=4,         # estaciones: winter, spring, summer, autumn
    start_year=2008,   # inicio Landsat "moderno"
    end_year=2011,     # OJO: exclusivo → llega hasta 2024
    cloud_filter=True,
    max_cloud_cover=10
)

# Colección: 1 imagen por año, con bandas ['winter','spring','summer','autumn']
coll = proc.get_year_composite()

# Máximo interanual SOLO de la banda 'winter'
winter_max_alltime = coll.select('winter').median()#.rename('MNDWI_winter_max_alltime')

winter_p95_alltime = coll.select('winter') \
    .reduce(ee.Reducer.percentile([95])) \
    .rename('MNDWI_winter_p95_alltime')

# Visualizar
viz = {'min': -0.2, 'max': 0.6}
Map.addLayer(winter_p95_alltime, viz, 'Landsat MNDWI Winter MAX (All-time)')

There we go again...
Loading Landsat WRS-2 geometry from GitHub...
Found Landsat tile for Path 202, Row 34
Applying cloud filter to Landsat: max 10% cloud cover
Using MODIS Terra + Aqua LST (maximum coverage)
Using all Sentinel-1 orbits (ascending + descending).
Applying S1 ARD preprocessing:
  - Speckle filter: REFINED_LEE
  - Terrain correction: True
  - Terrain model: VOLUME
Landsat collection includes thermal bands: ST_B10 (L8/9) and ST_B6 (L4/5/7)
Landsat collection configured with cloud filtering
Year 2008: Successfully processed 4 periods using mndwi index
Year 2009: Successfully processed 4 periods using mndwi index
Year 2010: Successfully processed 4 periods using mndwi index


## Export

#### Download local

In [ ]:
proc.get_export_single(
    image=img,
    filename="winter2020_ndvi.tif",
    scale=30,
    crs="EPSG:32630",
    region=proc.roi
)

#### To Drive

##### we need a new authenticate for that

In [ ]:
import ee, os, pathlib, shutil

# 1) Borra credenciales para forzar reauth
cred_path = pathlib.Path.home() / ".config" / "earthengine" / "credentials"
if cred_path.exists():
    shutil.rmtree(cred_path.parent)  # borra la carpeta earthengine completa

# 2) Autentica con scopes (usa la UI del notebook)
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/drive',
    ]
)

# 3) Inicializa
ee.Initialize(project='ee-digdgeografo')

In [ ]:
task = proc.export_to_drive(
    image=winter_p95_alltime,
    description="mndwi_winter_p95_alltime",
    folder="ndvi2gif_exports",
    file_format="GeoTIFF",
    crs="EPSG:32629",
    scale=30,
    max_pixels=int(1e13),
    file_dimensions=8192,    # opcional para trocear
    clip_region=True        # opcional para recortar
    #format_options={"cloudOptimized": True},  # <-- sin 'compression'
)
print("Task:", task.id)

#### To asset

In [9]:
task = proc.export_to_asset(
    image=winter_p95_alltime,  # MNDWI = float
    asset_id="users/ee-digdgeografo/assets/mndwi_winter_p95_alltime",
    description="mndwi_winter_p95_alltime",
    region=proc.roi,
    crs="EPSG:32630",
    scale=30,
    pyramiding_policy={"MNDWI_winter_p95_alltime": "mean"},  # <- float
    clip_region=True,
    overwrite=True
)

In [ ]:
## Probando LST
# Test 1: LST con Landsat (debería detectar automáticamente L4/5/7/8/9)
lst_landsat_20_25 = NdviSeasonality(roi=roi, sat='Landsat', key='percentile', percentile=95, index='lst', start_year=2020, end_year=2025)
landsat_2025_collection = lst_landsat_20_25.get_year_composite()

lst_landsat_15_20 = NdviSeasonality(roi=roi, sat='Landsat', key='percentile', percentile=95, index='lst', start_year=2015, end_year=2020)
landsat_1520_collection = lst_landsat_15_20.get_year_composite()

lst_landsat_95_00 = NdviSeasonality(roi=roi, sat='Landsat', key='percentile', percentile=95, index='lst', start_year=1995, end_year=2000)
landsat_9500_collection = lst_landsat_95_00.get_year_composite()

vizParams = {'bands': ['winter', 'spring', 'summer'], 'min': 5, 'max':35}

Map.addLayer(landsat_2025_collection.median(), vizParams, 'Landsat 20-25 LST')
Map.addLayer(landsat_1520_collection.median(), vizParams, 'Landsat 15-20 LST')
Map.addLayer(landsat_9500_collection.median(), vizParams, 'Landsat 95-00 LST')

In [ ]:
# Probando LST
# Test 2: LST con MODIS 

lst_modis_1011 = NdviSeasonality(roi=roi, sat='MODIS', index='lst', start_year=2001, end_year=2005)
modis_collection_1011 = lst_modis_1011.get_year_composite()

lst_modis_1122 = NdviSeasonality(roi=roi, sat='MODIS', index='lst', start_year=2020, end_year=2025)
modis_collection_1122 = lst_modis_1122.get_year_composite()

vizParams = {'bands': ['winter', 'spring', 'summer'], 'min': 15, 'max':45}

Map.addLayer(modis_collection_1011.median(), vizParams, 'MODIS 1011 LST')
Map.addLayer(modis_collection_1122.median(), vizParams, 'MODIS 1122 LST')

In [ ]:
# 1. Convertir Feature a Geometry
roi_geom = roi.geometry() if hasattr(roi, 'geometry') else roi

# 2. Ahora sí funciona el diagnóstico
lst_early = NdviSeasonality(roi=roi, sat='MODIS', index='lst', start_year=2001, end_year=2002)
early_collection = lst_early.get_year_composite()

first_image = early_collection.first()
stats = first_image.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi_geom,  # ← Usar roi_geom en lugar de roi
    scale=1000,
    maxPixels=1e9
).getInfo()

print("Estadísticas reales de LST 2001:", stats)

# 3. Crear visualización con rangos automáticos
if stats:
    winter_min = stats.get('winter_min', -10)
    winter_max = stats.get('winter_max', 40)
    
    vizParams_auto = {
        'bands': ['winter', 'spring', 'summer'], 
        'min': winter_min,
        'max': winter_max,
        'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
    }
    
    print(f"Rango real: {winter_min:.1f} a {winter_max:.1f}°C")
    Map.addLayer(first_image, vizParams_auto, 'MODIS 2001 LST (auto-range)')

# 4. También probar con rango amplio
vizParams_wide = {
    'bands': ['winter', 'spring', 'summer'], 
    'min': -30,
    'max': 50,
    'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
}
Map.addLayer(first_image, vizParams_wide, 'MODIS 2001 LST (wide range)')

In [ ]:
## Nuevos índices

# 2. UTFVI - Islas de calor urbano (requiere sensor con LST)
urban_heat = NdviSeasonality(
    roi=roi, 
    sat='Landsat',  # Tiene LST
    index='utfvi', 
    periods=4,
    start_year=2020, 
    end_year=2023
)
utfvi_collection = urban_heat.get_year_composite()

# Visualización UTFVI
utfvi_viz = {
    'bands': ['summer'],  # Verano para máximo efecto isla de calor
    'min': -0.01, 'max': 0.05,
    'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
}
Map.addLayer(utfvi_collection.select('summer'), utfvi_viz, 'Urban Heat Island')

In [ ]:
vci_analysis = NdviSeasonality(
    roi=roi, 
    sat='S2', 
    index='vci', 
    periods=12,  # Mensual para ver evolución de sequía
    start_year=2020, 
    end_year=2023
)
vci_collection = vci_analysis.get_year_composite()

# Visualización VCI
vci_viz = {
    'bands': ['january', 'april', 'july'],
    'min': 0, 'max': 100}

Map.addLayer(vci_collection.mean(), vci_viz, 'VCI Analysis')

In [ ]:
# 4. WI2015 - Mapeo avanzado de agua
water_mapping = NdviSeasonality(
    roi=roi, 
    sat='Landsat',
    key='percentile',
    percentile=15,
    index='wi2015', 
    periods=12,
    start_year=2023, 
    end_year=2025
)
wi2015_collection = water_mapping.get_year_composite()

# Visualización WI2015
wi2015_viz = {
    'bands': ['january', 'march', 'may'],
    'min': -1, 'max': 2,
    #'palette': ['brown', 'yellow', 'lightblue', 'blue', 'darkblue']
}
Map.addLayer(wi2015_collection.mean(), wi2015_viz, 'Advanced Water Mapping')

In [ ]:
# 5. NDBI - Crecimiento urbano
urban_growth = NdviSeasonality(
    roi=roi, 
    sat='Landsat', 
    index='ndbi', 
    periods=4,
    start_year=2010, 
    end_year=2023
)
ndbi_collection = urban_growth.get_year_composite()

# Visualización NDBI
ndbi_viz = {
    'bands': ['winter', 'spring', 'summer'],
    'min': -0.3, 'max': 0.5,
    #'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
}
Map.addLayer(ndbi_collection.mean(), ndbi_viz, 'Urban Development')

In [ ]:
# 3. NBR - Detección de incendios
fire_analysis = NdviSeasonality(
    roi=roi, 
    sat='Landsat',
    key='percentile',
    percentile=10,
    index='nbr', 
    periods=4,  # Quincenal para captar eventos de fuego
    start_year=2016, 
    end_year=2019
)
nbr_collection = fire_analysis.get_year_composite()

# Visualización NBR
nbr_viz = {
    'bands': ['summer', 'spring', 'winter'],  # Período de incendios (ej: junio-julio)
    'min': -0.5, 'max': 0.5,
    #'palette': ['red', 'orange', 'yellow', 'lightgreen', 'darkgreen']
}
Map.addLayer(nbr_collection.median(), nbr_viz, 'Burn Analysis')

In [ ]:
Map = geemap.Map()
Map

In [ ]:
# Test rápido
from ndvi2gif import NdviSeasonality

# Test S3
s1_test = NdviSeasonality(sat='S1', index='vv')
print(s1_test.get_available_indices('S1'))

In [ ]:
# Montaña Palentina
pol = '/media/diego/Datos4/EBD/Consultas/Fpalomares/sureste.shp'
myclass = NdviSeasonality(roi=pol, sat='S1', 
                          key='percentile', percentile=85, periods=12, start_year=2019, end_year=2020, index='vv', 
                          normalize_sar=False, sar_speckle_filter='REFINED_LEE', sar_terrain_model='VOLUME')
Landsat = myclass.get_year_composite()
vizParams = {'bands': ['january', 'may', 'july'], 'min': -17, 'max':-15}

Map.addLayer(Landsat, vizParams, 'S1 VSDI P85 2019')

In [ ]:
# Montaña Palentina
pol = '/media/diego/Datos4/EBD/Consultas/Fpalomares/sureste.shp'
myclass = NdviSeasonality(roi=pol, sat='S2', 
                          key='percentile', percentile= 85, periods=12,start_year=2019, end_year=2020, index='ndvi')
Landsat = myclass.get_year_composite()
vizParams = {'bands': ['april', 'may', 'june'], 'min': 0.2, 'max':0.8}
#vizParams = {'bands': ['p2', 'p10', 'p13'], 'min': 0.1, 'max':0.8}

Map.addLayer(Landsat, vizParams, 'NDVI P80 S2 2020-25')

## Estadísticas

In [ ]:
# Prueba manual CORREGIDA
print("=== PRUEBA MANUAL CORREGIDA ===")

# 1. Cargar el shapefile
roi = geemap.shp_to_ee(polygons_path)
print("Shapefile cargado, número de polígonos:", roi.size().getInfo())

# 2. Tomar una imagen (primer año)
first_image = ee.Image(Landsat.toList(Landsat.size()).get(0))
print("Bandas de la imagen:", first_image.bandNames().getInfo())

# 3. Calcular estadísticas CORRECTAMENTE (imagen.reduceRegions, no roi.reduceRegions)
stats = first_image.reduceRegions(
    collection=roi,  # Nota: collection es el FeatureCollection
    reducer=ee.Reducer.median(),
    scale=10
)

print("Estadísticas calculadas, número de resultados:", stats.size().getInfo())

# 4. Convertir a GeoDataFrame
gdf = geemap.ee_to_gdf(stats)
print("Columnas del resultado:", gdf.columns.tolist())
print("Forma del DataFrame:", gdf.shape)
print("Primeras filas:")
print(gdf.head())

In [ ]:
gdf.plot()

In [ ]:
# Ruta a tus polígonos
polygons_path = '/media/diego/Datos4/EBD/Consultas/Fpalomares/parcelas_sureste.shp'

# Ahora debería funcionar correctamente
all_stats = myclass.get_stats(
    geom=polygons_path,
    name='ndvi_monthly_stats',
    stat='MEDIAN',
    scale=10,
    to_file=True
)

# Ver qué años están disponibles
print("Años procesados:", list(all_stats.keys()))

# Acceder a los datos
year_2020 = all_stats[2020]  # O el año que esté disponible
print("Columnas disponibles:", year_2020.columns.tolist())

In [ ]:
# 1. Crear la instancia con percentil 90
myclass = NdviSeasonality(roi=pol, sat='S2', 
                          key='percentile', percentile= 90, periods=12,start_year=2018, end_year=2023, index='ndvi')
# 2. Obtener toda la colección de imágenes (una por año)
collection = myclass.get_year_composite()

# 3. Calcular la mediana de todos esos percentiles 90
median_of_p90 = collection.median()

# 4. Exportar esa imagen única
myclass.get_export_single(median_of_p90, 'ndvi_median_of_p90_2018-2023.tif')

In [ ]:
myclass.get_export(scale=20)

In [ ]:
# Here we select ndwi max for 12 periods seasons in Andalucia
myclass = NdviSeasonality(roi=roi, sat='S2', key='perc_95', periods=12,start_year=2016, end_year=2022, index='ndwi')
s2 = myclass.get_year_composite().median()

vizParams = {'bands': ['january', 'april', 'september'], 'min': 0, 'max':0.9}

Map.addLayer(s2, vizParams, 'Sentinel 2 Andalucia ndwi 4')

In [ ]:
# Doñana NDVI 
# Here we select ndwi max for 12 periods seasons
# Show graphics and pixel values
# Also change to 24 periods

myclass = NdviSeasonality(roi=roi, sat='S2', key='perc_90', periods=12,start_year=2019, end_year=2022, index='ndvi')
s2 = myclass.get_year_composite()

vizParams = {'bands': ['january', 'july', 'october'], 'min': 0, 'max':0.8}

Map.addLayer(s2, vizParams, 'P90 4')

Of course, it's possible to use loops to load several data at once

In [ ]:
Map = geemap.Map()
Map

In [ ]:
roi = Map.draw_last_feature

In [ ]:
# Statistics loop

stats = ['max', 'median', 'mean', 'perc_90', 'perc_95']
vizParams = {'bands': ['march', 'april', 'june'], 'min': 0, 'max':0.8}

for i in stats:

    myclass = NdviSeasonality(roi=roi, sat='S2', key=i, periods=12,start_year=2020, end_year=2022, index='ndvi')
    s2 = myclass.get_year_composite()
    Map.addLayer(s2, vizParams, str(i))

In [ ]:
# Years loop
# Change roi to Doñana deimsID

years = [i for i in range(2018, 2023)]

vizParams = {'bands': ['march', 'april', 'june'], 'min': 0, 'max':0.8}

for y in years:
    
    name = 'maxNDVI_' + str(y)
    myclass = NdviSeasonality(roi=roi, sat='S2', key='perc_90', periods=12,start_year=y, end_year=y+1, index='ndvi')
    s2 = myclass.get_year_composite()
    Map.addLayer(s2, vizParams, name)

We could do something bigger. 
Africa Example with MODIS

In [ ]:
Map = geemap.Map()
Map

In [ ]:
roi = Map.draw_last_feature

In [ ]:
import ee
africa = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq("wld_rgn", "Africa"))

In [ ]:
# Africa NDVI Mean
# Also show Nile delta with S2, take a sight on what you can see over the desert

Africa = NdviSeasonality(roi=roi, sat='S2', key='mean', periods=4,start_year=2020, end_year=2022, index='ndvi')
MODIS = Africa.get_year_composite()

vizParams = {'bands': ['summer', 'winter', 'spring'], 'min': 0.1, 'max':0.8}
Map.addLayer(MODIS, vizParams, 'S2_P95_24_s2_s2')

We said deimsPY allows to select your eLTER site qith only using your deimsID, so let's check it out

In [ ]:
Map = geemap.Map()
Map

In [ ]:
# Braila Island Flood

myclass = NdviSeasonality(roi='deimsid:https://deims.org/d4854af8-9d9f-42a2-af96-f1ed9cb25712', sat='S2', key='perc_95', periods=12,start_year=2017, end_year=2022, index='ndwi')
s2 = myclass.get_year_composite()
vizParams = {'bands': ['october', 'june', 'january'], 'min': 0.1, 'max':0.8}
Map.addLayer(s2, vizParams, 'Braila Island Median L')

Let's try to get the maximum flood in al januarys

In [ ]:
# Max january flood

januaryFloodmax = s2.select('january').max()
Map.addLayer(januaryFloodmax, {'min': 0, 'max':0.8}, 'January Max Flood P90')

In [ ]:
Map = geemap.Map()
Map

In [ ]:
roi = Map.draw_last_feature

In [ ]:
# Doñana NDVI Stats
myclass = NdviSeasonality(roi='deimsid:https://deims.org/bcbc866c-3f4f-47a8-bbbc-0a93df6de7b2', sat='S2', 
                          key='max', periods=12,start_year=2019, end_year=2020, index='ndvi')
Landsat = myclass.get_year_composite()
vizParams = {'bands': ['february', 'november', 'july'], 'min': 0.1, 'max':0.8}

Map.addLayer(Landsat, vizParams, 'Max')

We can also have zonal statistic

In [ ]:
# Points and polygons zonal stats

import os

# This is a Google Earth Engine Feature Collection
alcornoques = ee.FeatureCollection('users/digdgeografo/pajarera')

# This is a local point shapefile
samples = '/home/diego/samples.shp'
geom = geemap.shp_to_ee(samples)

# Define the ouptut shape
out_shp = os.path.join(os.getcwd(), 'my_samples_stats_2.shp')

# Compute the statistics
geemap.zonal_statistics(Landsat, in_zone_vector=geom, out_file_path=out_shp, statistics_type='MEAN', scale=30)

It is also possible to export the collections. There's a problem qith size limits from GEE but small areas are ok

In [ ]:
Map = geemap.Map()
Map

In [ ]:
# Take a smaller roi

roi = Map.draw_last_feature

In [ ]:
# Here we select ndwi max for 12 periods seasons in Andalucia

myclass = NdviSeasonality(roi=roi, sat='Landsat', key='max', periods=12,start_year=2017, end_year=2020, index='ndvi')
Landsat = myclass.get_year_composite()
vizParams = {'bands': ['january', 'april', 'september'], 'min': 0.1, 'max':0.8}

Map.addLayer(Landsat, vizParams, 'Max')

In [ ]:
# Export it and load the data in QGIS

myclass.get_export()

In [ ]:
Map = geemap.Map()
Map

In [ ]:
roi = Map.draw_last_feature

In [ ]:
# Last but not least and exaample of Sierra Nevada NDSI

myclass = NdviSeasonality(roi=roi, sat='S2', key='median',
                          periods=12,start_year=2020, end_year=2022, index='ndsi')
s2 = myclass.get_year_composite()

vizParams = {'bands': ['january', 'february', 'march'], 'min': 0, 'max':0.8}
Map.addLayer(s2, vizParams, 'S2_MAX_Sierra Nevada')